In [3]:
import json
import sys
import os
import time
from tqdm import tqdm
from openai import OpenAI
from striprtf.striprtf import rtf_to_text
from dotenv import load_dotenv
import importlib
import csv
from datasets import load_dataset

# Load environment variables from current directory .env and override existing ones
load_dotenv(".env", override=True)

# load api key from environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

# Get the model from environment variable
model = os.getenv("OPENAI_MODEL")
if not model:
    raise ValueError("OPENAI_MODEL not found in .env file")

# Get temperature and max_tokens from environment variables with defaults
temperature = float(os.getenv("OPENAI_TEMPERATURE", "0.7"))
max_tokens = int(os.getenv("OPENAI_MAX_TOKENS", "1000"))

#set client with key from .env
client = OpenAI(api_key=api_key)

In [ ]:
def gpt_generate(messages, model=model, temperature=temperature, max_tokens=max_tokens):

    max_retries = 5
    retries = 0

    while True:
        try:
            response = client.chat.completions.create(
                model=model, messages=messages, temperature=temperature, max_tokens=max_tokens
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"Error: {e}")
            retries += 1
            if retries >= max_retries:
                print("Max retries reached. Exiting.")
                sys.exit(1)
            wait_time = 2 ** retries
            print(f"Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
            continue

Load your original dataset from a file. Will need to have a 'prompt' item and some form of preference annotation for each prompt. 

In [10]:
dataset_filepath = '/Users/henrybell/Desktop/Research/Elicitation/Principle Elicitation Repo/Principle-Elicitation/princ_from_reasons/data_subset.json'

with open(dataset_filepath, 'r') as f:
    dataset = json.load(f)

print(dataset[0]['all_preferences_unprocessed'])

[{'strength': -1, 'justification': '@Response 1 is slightly better than @Response 2. Both both responses are correct, well written, and do a good job explaining a very complicated topic.\n\n-Verbosity - @Response 2 presents the same information in two ways, which is unnecessary and makes the response overly long.'}, {'strength': -1, 'justification': '@Response 1 edges out slightly over @Response 2 because it offers a broader view of potential applications and slightly more context on why nested genetic algorithms are beneficial, especially in terms of dealing with complex problems and the manual difficulty of determining parameters. This adds a bit more depth and utility to the explanation, making it slightly more comprehensive.'}, {'strength': -1, 'justification': '@Response 1 is better than @Response 2 because it is concise.\n\n\n\nVerbosity: @Response 2 is more verbose; it provides more details about nested genetic algorithms. @Response 1, on the other hand, provides a concise and s

Load the candidate principles. 

In [ ]:
principles_filepath = '/Users/henrybell/Desktop/Research/Elicitation/Principle Elicitation Repo/Principle-Elicitation/Principle Generations/10_23_individual_just._gpt/10_23_summarized_principles.json'

with open(principles_filepath, 'r') as f:
    principles = json.load(f)



Load prompts

In [ ]:
relevancy_prompt_path = '/Users/henrybell/Desktop/Research/Elicitation/Principle Elicitation Repo/Principle-Elicitation/Principle Scoring/Prompts/relevance_prompt.txt'

with open(relevancy_prompt_path, 'r') as f:
    relevancy_prompt_template = f.read()



Calculate relevance scores for each principle + prompt combo.

For every prompt, check the relevance of every principle. Score 1-5.